# XGBoost Model

This notebook trains an XGBoost model for cancellation prediction.

## Metrics Targets
- **F2-Score**: ≥ 0.68
- **Recall**: ≥ 70%
- **Precision**: ≥ 60%

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import (
    fbeta_score, precision_score, recall_score,
    average_precision_score, roc_auc_score, f1_score,
    confusion_matrix, precision_recall_curve, roc_curve
)
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Load processed data
data_dir = "../data/silver"

X_train = pd.read_parquet(os.path.join(data_dir, "X_train.parquet"))
X_val = pd.read_parquet(os.path.join(data_dir, "X_val.parquet"))
X_test = pd.read_parquet(os.path.join(data_dir, "X_test.parquet"))

y_train = pd.read_parquet(os.path.join(data_dir, "y_train.parquet"))['is_cancelled']
y_val = pd.read_parquet(os.path.join(data_dir, "y_val.parquet"))['is_cancelled']
y_test = pd.read_parquet(os.path.join(data_dir, "y_test.parquet"))['is_cancelled']

print(f"X_train shape: {X_train.shape}")
print(f"Features: {X_train.columns.tolist()}")

In [ ]:
# Helper functions
def evaluate_model(y_true, y_pred, y_prob, dataset_name=""):
    f2 = fbeta_score(y_true, y_pred, beta=2)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    pr_auc = average_precision_score(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob)
    
    print(f"\n{'='*60}")
    print(f"EVALUATION - {dataset_name}")
    print(f"{'='*60}")
    print(f"F2-Score:  {f2:.4f} {'✅' if f2 >= 0.68 else '❌'}")
    print(f"Recall:    {recall:.4f} {'✅' if recall >= 0.70 else '❌'}")
    print(f"Precision: {precision:.4f} {'✅' if precision >= 0.60 else '❌'}")
    print(f"PR-AUC:    {pr_auc:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    
    return {'f2': f2, 'f1': f1, 'precision': precision, 'recall': recall, 'pr_auc': pr_auc, 'roc_auc': roc_auc}

def find_optimal_threshold(y_true, y_prob, beta=2):
    thresholds = np.arange(0.1, 0.9, 0.01)
    best_threshold, best_f_beta = 0.5, 0
    for thresh in thresholds:
        y_pred = (y_prob >= thresh).astype(int)
        f_beta = fbeta_score(y_true, y_pred, beta=beta)
        if f_beta > best_f_beta:
            best_f_beta = f_beta
            best_threshold = thresh
    return best_threshold, best_f_beta

def plot_evaluation(y_true, y_pred, y_prob, title=""):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    axes[0].set_title(f'Confusion Matrix - {title}')
    
    # PR Curve
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    axes[1].plot(recall, precision, 'b-', linewidth=2, label=f'PR-AUC = {pr_auc:.3f}')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title('Precision-Recall Curve')
    axes[1].legend()
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob)
    axes[2].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC-AUC = {roc_auc:.3f}')
    axes[2].plot([0, 1], [0, 1], 'k--')
    axes[2].set_xlabel('False Positive Rate')
    axes[2].set_ylabel('True Positive Rate')
    axes[2].set_title('ROC Curve')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()

## Train XGBoost

In [ ]:
# Calculate scale_pos_weight for imbalanced data
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

# Train model
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='aucpr',
    early_stopping_rounds=20,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)
print(f"Model trained. Best iteration: {model.best_iteration}")

In [ ]:
# Get predictions
y_val_prob = model.predict_proba(X_val)[:, 1]
y_test_prob = model.predict_proba(X_test)[:, 1]

In [ ]:
# Find optimal threshold
optimal_threshold, best_f2 = find_optimal_threshold(y_val, y_val_prob)
print(f"Optimal threshold: {optimal_threshold:.2f}")
print(f"Best F2-Score: {best_f2:.4f}")

In [ ]:
# Evaluate on validation
y_val_pred = (y_val_prob >= optimal_threshold).astype(int)
val_metrics = evaluate_model(y_val, y_val_pred, y_val_prob, "Validation")
plot_evaluation(y_val, y_val_pred, y_val_prob, "Validation")

In [ ]:
# Final evaluation on TEST set
y_test_pred = (y_test_prob >= optimal_threshold).astype(int)
test_metrics = evaluate_model(y_test, y_test_pred, y_test_prob, "TEST SET")
plot_evaluation(y_test, y_test_pred, y_test_prob, "Test")

## Feature Importance

In [ ]:
# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance:")
print(feature_importance)

In [ ]:
# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(feature_importance['feature'], feature_importance['importance'])
ax.set_xlabel('Importance')
ax.set_title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()

## Save Model

In [ ]:
# Save model
model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)

model_artifacts = {
    'model': model,
    'optimal_threshold': optimal_threshold,
    'feature_names': X_train.columns.tolist(),
    'test_metrics': test_metrics
}

with open(os.path.join(model_dir, "xgboost.pkl"), 'wb') as f:
    pickle.dump(model_artifacts, f)

print(f"Model saved to {model_dir}/xgboost.pkl")